# QLoRA Planning Lab

## Goal / Mục tiêu
Lập data/model/VRAM plan trước khi mở notebook Unsloth có GPU.

## Setup / Chuẩn bị
Không tải model và không cần GPU; mọi con số là estimate cần xác minh bằng profiler.

In [ ]:
from collections import Counter

MODEL_B = 3.0
BITS = 4
WIDTH = 3072
RANK = 16
TARGET_MODULES = 64


## Steps / Các bước
### 1. Audit một dataset hội thoại mẫu

In [ ]:
records = [
    {'messages': [{'role': 'user', 'content': 'Tóm tắt văn bản'}, {'role': 'assistant', 'content': 'Bản tóm tắt'}], 'source': 'demo', 'license': 'mit'},
    {'messages': [{'role': 'user', 'content': 'Phân loại rủi ro'}, {'role': 'assistant', 'content': 'low'}], 'source': 'demo', 'license': 'mit'},
]
patterns = Counter(tuple(message['role'] for message in row['messages']) for row in records)
assert all(row['messages'][-1]['role'] == 'assistant' for row in records)
print({'rows': len(records), 'role_patterns': dict(patterns)})


### 2. Ước lượng weight-only memory và LoRA parameters

In [ ]:
weight_gib = MODEL_B * 1e9 * BITS / 8 / 2**30
lora_parameters = TARGET_MODULES * 2 * WIDTH * RANK
print({'weight_only_gib': round(weight_gib, 2), 'lora_parameters': lora_parameters})


## Checks / Kiểm tra

In [ ]:
assert weight_gib > 0
assert lora_parameters < MODEL_B * 1e9
assert set(patterns) == {('user', 'assistant')}
print('PASS — tiếp theo phải cộng activation/KV/optimizer/runtime overhead.')


## Next Steps / Bước tiếp
Chọn model/revision và notebook hiện hành từ tài liệu Unsloth; chạy smoke test 20–60 step trước full run.